## MATH

In [1]:
import os
os.environ['http_proxy'] = 'http://10.14.30.39:7897'
os.environ['https_proxy'] = 'http://10.14.30.39:7897'
os.environ['all_proxy'] = 'socks5://10.14.30.39:7897'
os.environ['no_proxy'] = 'localhost,127.0.0.1'

In [2]:
from datasets import load_dataset
ds = load_dataset("smolagents/codeagent-traces")

/root/anaconda3/envs/agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds_filtered = ds.filter(lambda example: example['is_correct'] == True and example['is_thinking'] == False)

In [4]:
math = ds_filtered['filtered'].filter(lambda example: example['source'] == 'MATH')

In [5]:
math_traces = math.train_test_split(test_size=0.1, seed=42)

In [6]:
math_traces

DatasetDict({
    train: Dataset({
        features: ['model_id', 'source', 'original_question', 'messages', 'true_answer', 'prediction', 'is_correct', 'is_thinking'],
        num_rows: 924
    })
    test: Dataset({
        features: ['model_id', 'source', 'original_question', 'messages', 'true_answer', 'prediction', 'is_correct', 'is_thinking'],
        num_rows: 103
    })
})

In [7]:
math_traces.save_to_disk("math_traces")

Saving the dataset (1/1 shards): 100%|██████████| 103/103 [00:00<00:00, 18850.39 examples/s]


## SimpleQA

In [8]:
simpleqa = ds_filtered['filtered'].filter(lambda example: example['source'] == 'SimpleQA')

In [9]:
simpleqa

Dataset({
    features: ['model_id', 'source', 'original_question', 'messages', 'true_answer', 'prediction', 'is_correct', 'is_thinking'],
    num_rows: 5028
})

In [10]:
# filter filter_year=0
def exist_uncorrect_filter(messages):
    assis_contents = ""
    for msg in messages:
        if msg['role'] == "assistant" and 'filter_year=0' in msg['content']:
            return True
    return False

simpleqa = simpleqa.filter(lambda example: not exist_uncorrect_filter(example['messages']))

In [11]:
simpleqa

Dataset({
    features: ['model_id', 'source', 'original_question', 'messages', 'true_answer', 'prediction', 'is_correct', 'is_thinking'],
    num_rows: 4840
})

In [12]:
# Filter out samples with token length > 15000
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def filter_by_token_length(example, max_length=15000):
    try:
        # Apply chat template to get the formatted text
        text = tokenizer.apply_chat_template(example['messages'], tokenize=False)
        tokens = tokenizer.tokenize(text)
        return len(tokens) <= max_length
    except:
        # If there's any error, keep the sample (conservative approach)
        return True

print(f"Before length filtering: {len(simpleqa)} samples")
simpleqa = simpleqa.filter(filter_by_token_length)
print(f"After length filtering: {len(simpleqa)} samples")
print(f"Retained: {len(simpleqa)/4840*100:.2f}% of original samples")

Before length filtering: 4840 samples
After length filtering: 3798 samples
Retained: 78.47% of original samples


In [13]:
simpleqa

Dataset({
    features: ['model_id', 'source', 'original_question', 'messages', 'true_answer', 'prediction', 'is_correct', 'is_thinking'],
    num_rows: 3798
})

In [14]:
simpleqa = simpleqa.shuffle(seed=42)
simpleqa_train = simpleqa.select(range(1800))
simpleqa_test = simpleqa.select(range(1800, len(simpleqa)))

from datasets import DatasetDict
simpleqa_traces = DatasetDict({
    'train': simpleqa_train,
    'test': simpleqa_test
})

In [15]:
simpleqa_traces.save_to_disk("sqa_traces")

Saving the dataset (1/1 shards): 100%|██████████| 1998/1998 [00:00<00:00, 39753.23 examples/s]
